# 3. The flag line and the errors

## What you will learn

- What the "flag line" is (the top 10% of an earlier year's scores), and why it comes from an earlier year.
- How to read the four outcomes of a flag: caught downgrades, missed downgrades, false alarms, and quiet and correct cases (not flagged, not downgraded).
- Why a flag is a reason to look at a company, not a reason to sell it.

## What you need

- Notebooks 1 and 2 (the panel, the label, the year-by-year test, ROC-AUC).
- Python with the packages in `model/requirements.txt` and `jupyter`. No licensed data and no internet.
- About half a minute to run (15 to 24 seconds on one 16-core laptop; older laptops are slower). The temporary example data is deleted in the last cell.

## Real data

These notebooks use made-up example data, so they run anywhere in seconds. To work with the real data,
see notebook 5 (`05_real_data.ipynb`).

> **The example data is made up.** Every company here is called "Example Co ...", and every number comes from a random
> number generator. A result on this data says nothing about real companies. The example is only there so that all the
> code runs on your own computer. Do not compare its numbers with the lecture's real results as if they measured the same thing.

## Set up

As before: find the code, make a temporary folder and an example table, and run the year-by-year test for the main model (the logit)
with the package's command `python -m tcf_credit.evaluate`. The test writes every test score into `predictions.csv`.

In [ ]:
import os, sys, shutil, subprocess, tempfile
from pathlib import Path

import numpy as np
import pandas as pd

# Find the repository folder (the one that holds model/src/tcf_credit). This works whether the notebook
# is started from the repository folder or from model/notebooks.
here = Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p / "model" / "src" / "tcf_credit").is_dir())
SRC, SYN = root / "model" / "src", root / "model" / "synthetic"
sys.path[:0] = [str(SRC), str(SYN)]
# Commands that we start from the notebook need the package on their path too.
os.environ["PYTHONPATH"] = os.pathsep.join([str(SRC)] + ([os.environ["PYTHONPATH"]] if os.environ.get("PYTHONPATH") else []))

work = Path(tempfile.mkdtemp(prefix="tcf_nb3_"))   # a temporary folder; the last cell deletes it
pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 40)
print("repository:", root)

In [ ]:
panel_path = work / "example_panel.csv.gz"
done = subprocess.run([sys.executable, str(SYN / "make_panel.py"), "--out", str(panel_path), "--firms", "400", "--seed", "1"],
                      check=True, capture_output=True, text=True)
print(done.stdout.strip())

In [ ]:
out_dir = work / "run"
res = subprocess.run([sys.executable, "-m", "tcf_credit.evaluate", str(panel_path), str(out_dir),
                      "--first", "2016", "--last", "2024", "--only", "logit"], capture_output=True, text=True)
if res.returncode != 0:
    print(res.stderr[-1500:])
assert res.returncode == 0
pred = pd.read_csv(out_dir / "predictions.csv")
pooled = pd.read_csv(out_dir / "pooled.csv")
print(len(pred), "test scores, one per test report, for test years", pred.year.min(), "to", pred.year.max())
pred.head(4)

## What is a flag, and where does the line come from?

A probability is not yet a decision. To act, we need a **flag line**: a score above which we say "look at this company".
The model uses the following rule, set once in `evaluate.py`:

> The flag line for test year **Y** is the score that only the top **10%** of reports of year **Y-2** exceed. Those reports are
> scored by a model trained on years up to **Y-4**, so they are scored "out of sample" (by a model that never saw them), the way the test year will be.
> A report in year Y is **flagged** if its probability is at or above that line.

**Why from an earlier year and not from year Y itself?** If we took the top 10% of year Y's own scores, we would need to know
all of year Y's scores before flagging any single company. In real life you receive the reports one at a time during the year.
Fixing the line in advance, from a year that is already finished, is something you could really do. A side effect: the share flagged
is not exactly 10% each year. It goes up when the scores are higher than in the earlier year and down when they are lower
(on the lecture's real data it was 6.6% in 2020).
(The command also writes a second, "rank" rule that flags about 10% of each test year. We keep it only to compare.)

The next cell rebuilds the flag line of one test year with the package's own functions and checks it against the command's output.

In [ ]:
from tcf_credit import evaluate as E
from tcf_credit.features_v2 import V2_COLUMNS

E.EXTRA[:] = V2_COLUMNS + E.MARKET
E.ONLY[:] = ["logit"]
data = pd.read_csv(panel_path)
data = E.fixed_order(data)        # the same fixed row order as evaluate.main(), so the fits match the command
data = data[data.has_prices == 1].copy()
data["year"] = pd.to_datetime(data["t"]).dt.year
data["ebit_interest"] = data["ebit_interest"].clip(-50, 100)

params = E.tune(data, first=2016, end=2014)
make, cols_, impute, sectors = E.models(params)["logit"]

Y = 2020
train, test = data[data.year <= Y - 2], data[data.year == Y]
calib_train, calib = data[data.year <= Y - 4], data[data.year == Y - 2]     # the earlier year that sets the line

_, p_test = E.fit_predict(make, cols_, impute, sectors, train, test)          # model for year Y, scores year Y
_, p_calib = E.fit_predict(make, cols_, impute, sectors, calib_train, calib)  # older model, scores year Y-2
line = np.quantile(p_calib, 1 - 0.10)
flagged = p_test >= line

print(f"test year {Y}: the line is {line:.3f}; it is the score that 10% of the {len(calib)} reports of {Y-2} reach or exceed.")
print(f"{flagged.sum()} of {len(test)} reports in {Y} are flagged ({flagged.mean():.1%}).")

# A check that can fail: the same flags as the command wrote.
cmd = pred[(pred.model == "logit") & (pred.year == Y)].reset_index(drop=True)
assert (cmd.flag_cut.to_numpy() == flagged).all(), "hand-made flags differ from the command"
print("same flags as predictions.csv: yes")

## Flags year by year

The table shows, for each test year, the share of reports flagged by the fixed line ("earlier-year line") and by the rank rule (about 10% of each year).

In [ ]:
lg = pred[pred.model == "logit"]
by_year = lg.groupby("year").agg(reports=("label", "size"), downgrades=("label", "sum"),
                                 flagged_by_line=("flag_cut", "mean"), flagged_by_rank=("flag_rank", "mean")).round(3)
by_year

## Caught, missed, false alarms: the 2x2 table

Every test report ends in one of four boxes:

|  | Downgraded within 12 months | Not downgraded |
|---|---|---|
| **Flagged** | caught (true positive) | false alarm (false positive) |
| **Not flagged** | missed (false negative) | quiet and correct (true negative) |

Two rates matter and they have different denominators:

- **Share of downgrades caught** = caught / (caught + missed). It is about the downgrades.
- **False-alarm rate** = false alarms / (all reports that were not downgraded). It is about the companies that did fine.

In [ ]:
c = pooled[(pooled.model == "logit") & (pooled.rule == "calibrated_cut")].iloc[0]
table = pd.DataFrame([[c.TP, c.FP], [c.FN, c.TN]],
                     index=["Flagged", "Not flagged"], columns=["Downgraded within 12 months", "Not downgraded"]).astype(int)
table.loc["Total"] = table.sum()
table["Total"] = table.sum(axis=1)
print(table.to_string())
print()
print(f"share of downgrades caught: {c.TP}/{c.TP + c.FN} = {c.TP / (c.TP + c.FN):.1%}   (missed: {c.miss_rate:.1%})")
print(f"false-alarm rate: {c.FP}/{c.FP + c.TN} = {c.false_alarm_rate:.1%}")
print(f"share of flags that were real downgrades (precision): {c.TP}/{c.flagged} = {c.precision:.1%}; share of all reports that were downgraded: {c.base_rate:.1%}")

These are numbers for made-up data. For the lecture's real data, the flag line caught **407 of the 862** downgrades (so 455 were missed)
with **8.7%** false alarms. Reading that: the line finds about half of the downgrades in advance, and about 9 in 100
healthy companies get flagged anyway. The model is an early-warning light, not a crystal ball.

Moving the line trades the two errors against each other. Flag more companies and you catch more downgrades but raise more false alarms.

**Two flag rules, side by side.** `predictions.csv` holds two flags for every report, and they give different counts on purpose:

| Rule | How a report is flagged | Used for |
|---|---|---|
| **Earlier-year line** (`flag_cut`; the 2x2 table above) | its score is at or above a line fixed from an earlier year's scores, so the share flagged changes from year to year | the lecture's results; this is what could be done in real time |
| **Rank rule** (`flag_rank`; the table below) | it is among the riskiest share of that same test year, so about that share is flagged | only to show the trade-off; it needs the whole year's scores first |

So the "10%" row below does not equal the 2x2 table above. The first row of the next table repeats the earlier-year line so that you can compare the two directly.

In [ ]:
n_reports, n_down = len(lg), int(lg.label.sum())
rows = []
# first row: the earlier-year line (the 2x2 table above)
tp = int((lg.flag_cut & (lg.label == 1)).sum()); fp = int((lg.flag_cut & (lg.label == 0)).sum())
rows.append({"rule": "earlier-year line", "share flagged": f"{lg.flag_cut.mean():.1%}", "caught": tp, "missed": n_down - tp,
             "share of downgrades caught": round(tp / n_down, 3), "false alarms": fp,
             "false-alarm rate": round(fp / (n_reports - n_down), 3)})
for share in [0.05, 0.10, 0.20, 0.30]:
    flag = lg.groupby("year").p.transform(lambda s: s >= np.quantile(s, 1 - share))
    tp = int((flag & (lg.label == 1)).sum()); fn = int((~flag & (lg.label == 1)).sum())
    fp = int((flag & (lg.label == 0)).sum()); tn = int((~flag & (lg.label == 0)).sum())
    rows.append({"rule": "rank rule", "share flagged": f"{share:.0%}", "caught": tp, "missed": fn,
                 "share of downgrades caught": round(tp / (tp + fn), 3), "false alarms": fp,
                 "false-alarm rate": round(fp / (fp + tn), 3)})
pd.DataFrame(rows).set_index(["rule", "share flagged"])

## The level of the probabilities in a shock year

A model can **rank** companies well and still get the **level** of the probabilities wrong in a year when something hits everybody at once.
On the lecture's real data, in 2020 (the pandemic year) about **29%** of the reports were followed by a downgrade, while the model's
average predicted probability was only about **14%**. The ranking can still be good (that is what ROC-AUC measures), but the probabilities were too low for that
year. The flag line for 2020 was fixed from the scores of 2018, when the model's average probability was about 23%. Against that line the low
2020 probabilities rarely reached it: the line flagged only 46 of the 695 reports (6.6%), fewer than the 10% it was set for, and caught only 39 of the 203
downgrades. Probabilities that are too low put fewer companies above a line fixed earlier, not more.

The table compares, for each test year of the example data, the actual downgrade rate with the average predicted probability, and shows the ROC-AUC and the
share flagged. The example economy has only a mild cycle and no sudden shock like 2020, so expect the first two columns to stay fairly close. That is built in, not a finding.

In [ ]:
from sklearn.metrics import roc_auc_score
level = lg.groupby("year").apply(lambda g: pd.Series({"actual downgrade rate": g.label.mean(),
                                                     "average predicted probability": g.p.mean(),
                                                     "ROC-AUC": roc_auc_score(g.label, g.p),
                                                     "share flagged (earlier-year line)": g.flag_cut.mean()}), include_groups=False).round(3)
level

## Look at the errors

Numbers hide the cases. Here are the made-up companies behind the errors for the example data: the downgrades the model **missed**
(lowest scores among downgraded reports) and the **false alarms** (highest scores among reports without a downgrade).
The rating state is shown next to each one, so you can see whether the rating had already moved (a negative outlook or CreditWatch) before the event.
Our reading, not tested: a model that leans on the rating has trouble with companies whose rating had not yet moved.

In [ ]:
panel = pd.read_csv(panel_path)
look = lg.merge(panel[["cik", "t", "sp_name", "rating", "outlook_neg", "watch_neg", "debt_assets", "ebit_assets"]], on=["cik", "t"], how="left")
show = ["sp_name", "t", "p", "rating", "outlook_neg", "watch_neg", "debt_assets", "ebit_assets"]
missed = look[(look.label == 1) & (~look.flag_cut)].sort_values("p").head(5)
alarms = look[(look.label == 0) & (look.flag_cut)].sort_values("p", ascending=False).head(5)
print("Downgrades the model missed (lowest scores):")
display(missed[show].round(3).reset_index(drop=True))
print("False alarms (highest scores without a downgrade):")
display(alarms[show].round(3).reset_index(drop=True))

Notice what the tables show. The missed downgrades sit at low scores: the inputs looked healthy, and the downgrade came as a surprise.
The false alarms look genuinely risky on paper (high debt, low profit, weak rating), but S&P did not act within a year.
In both cases the model did what it can do with public numbers on that day.

## A flag is a reason to look, not a reason to sell

Look again at the precision above. Even among flagged reports, a large share were **not** downgraded. A flag raises the chance
of a downgrade a lot (compare precision with the base rate, the share of all reports that were downgraded), but it never makes the downgrade certain. And a downgrade by
S&P is not the same as a falling share price or a default.

So what should a flag lead to? **A closer look.** Read the company's latest 10-K and the rating agency's outlook,
check whether the bad numbers are one-off or lasting, and compare with similar companies. Then decide.
A missing flag does not mean "safe" either: about half of the downgrades were missed on the lecture's data.
The `credit-watch` agent in this repository does the first part of that look for you: it collects the ratings text from the filings and cites it.

**What the lecture found on the real (licensed) data, for comparison.** Test years 2016-2024, 6,380 annual reports, of which
862 were followed by an S&P downgrade within 12 months (13.5%).

| Result (real data) | Value |
|---|---|
| Logit: ROC-AUC / PR-AUC | 0.802 / 0.494 |
| Other single engines (not the fly models): ROC-AUC | 0.785 to 0.808 (boosting 0.808; no single engine is clearly better than the logit) |
| Averages that include the logit: ROC-AUC | 0.817 (logit-boosting) and 0.813 (all non-fly models); both clearly above the logit |
| S&P rating state alone: ROC-AUC | 0.757 |
| Fly brain models: ROC-AUC (explained in notebook 4) | 0.759 to 0.807 (real wiring had no consistent edge over shuffled wiring) |
| Flag line (top 10% of an earlier year's scores) | caught 407 of 862 downgrades, with 8.7% false alarms |

## Tidy up

The last cell deletes the temporary folder.

## Check yourself

1. For test year 2020, from which year's scores is the flag line taken, and why not from 2020 itself?

<details><summary>Answer</summary>

From 2018, scored by a model trained on years up to 2016. Taking the line from 2020's own scores would need all of 2020's scores before flagging any company, which you do not have in real time.
</details>

2. On the lecture's data, about 29% of the 2020 reports were downgraded, but the model's average probability for 2020 was about 14%. Does that mean the model ranked companies badly?

<details><summary>Answer</summary>

Not necessarily. ROC-AUC measures the ranking, and a ranking can stay good when the whole level is off. In a shock year the probabilities are too low on average (here about half of the actual rate), so read them as
relative risk, and expect fewer companies than usual to cross a line fixed from an earlier year: in 2020 the line flagged only 46 of 695 reports (6.6%)
and caught 39 of the 203 downgrades. (The missed share over all test years, 455 of the 862 downgrades, is the other half of the picture.)
</details>

3. A company is flagged. Which is the better next step: sell, or look closer? Give one reason.

<details><summary>Answer</summary>

Look closer. Even flagged reports are often not followed by a downgrade (false alarms), and a downgrade is not the same as a loss. The flag tells you where to spend your attention.
</details>

In [ ]:
shutil.rmtree(work, ignore_errors=True)
print("temporary folder deleted:", not work.exists())